<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_04_sequence_model_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 04 — Sequences: a Recurrent Network and an LSTM

**Deep Learning for Engineering · Aalborg University · Part 1**

L5.2 introduces two networks for sequences: the recurrent network and the LSTM.
This notebook builds both on one small forecasting problem and compares them with
a forecast that needs no model at all.

The problem: **one-step-ahead forecasting of substation demand.** Given the last
twenty-four hourly readings, predict the next one. Forty days of synthetic data,
generated on your machine.

The path is short:

1. load the data and cut it into 24-hour windows;
2. the **persistence** forecast — the next hour equals this hour;
3. the **recurrent network** and the **LSTM**, each defined in one cell, with
   every size and parameter count written beside the line that sets it;
4. train both with the same recipe;
5. one table and one plot.

The two networks use PyTorch's `nn.RNN` and `nn.LSTM`. The equations are in the
text above each cell, and the comments say which weights belong to which term.

---

## 0 · Setup and the data

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_5_core as core
core.keep_outputs()


In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

core.set_seed(0)

series = core.load_profile(n_days=40, seed=21)               # 40 days x 24 hours, demand in p.u.
X, y = core.make_windows(series, window=24, horizon=1)       # input: 24 hours; target: the next hour
X_train, y_train, X_test, y_test = core.split_series(X, y, frac=0.75)   # first 75 % trains, in time order

print("series  :", series.shape, " (40 days x 24 hours)")
print("windows :", X.shape, " targets:", y.shape)
print("training:", X_train.shape[0], " held out:", X_test.shape[0])

core.plot_series(series[:14 * 24], title="The first two weeks")
plt.show()

**What you should see.** `series : (960,)`, `windows : (936, 24, 1)`,
`targets: (936, 1)`, 702 training windows and 234 held out, and a trace with a
clear daily rhythm and two lighter days at each weekend.

Each window is a `(24, 1)` array: twenty-four hours, one value per hour. The
target is the hour after the window.

**The split is in time order.** `core.split_series` takes the first 75 % of the
windows for training and the last 25 % for testing, without shuffling. Two
neighbouring windows share twenty-three of their twenty-four hours, so a shuffled
split would put near-copies of the test windows in the training set, and the
held-out error would mean nothing.

---

## 1 · The baseline: persistence

**Persistence** predicts that the next hour equals this hour. It is the forecast
an operator makes without a computer, it has nothing to train, and over one hour
it is hard to beat. A network that does not beat it has learned nothing useful,
so every error later in this notebook is read against it.

It is written below as a model with the same input and output as the two networks,
so that all three are scored the same way.

### Your turn

In [ ]:
# TODO 1 --- the persistence model -----------------------------------------------------------
# One `...` to replace, in forward:
#   x[:, -1, :]        the last hour of each window, shape (batch, 1)
#
# window length : 24 hours are handed in, only the last one is read
# input size    : 1   (one demand value per hour, in p.u.)
# output size   : 1   (the demand one hour ahead, in p.u.)
# parameters    : 0   (nothing to learn, nothing to train)
class Persistence(nn.Module):
    def forward(self, x):                       # x is (batch, 24, 1)
        return x[:, -1, :]
# ------------------------------------------------------------------------------

persistence = Persistence()
with torch.no_grad():
    pred_persistence = persistence(torch.tensor(X_test)).numpy()
mse_persistence = core.mse(pred_persistence, y_test)

print(f"persistence parameters      : {core.count_parameters(persistence)}")
print(f"persistence held-out MSE    : {mse_persistence:.6f} p.u.^2")
print(f"persistence held-out RMSE   : {np.sqrt(mse_persistence):.4f} p.u.")
print(f"held-out demand, std. dev.  : {y_test.std():.4f} p.u.")

**What you should see.** `persistence parameters : 0`, a held-out MSE of
`0.003320`, an RMSE of `0.0576` p.u. and a standard deviation of the held-out
demand of `0.1560` p.u.

So an hour-old reading is already off by about six per cent of peak on average.
That is the number the two networks have to beat.

---

## 2 · The recurrent network

A recurrent network reads the window one hour at a time and carries a **hidden
state** $\mathbf{h}_t$ forward:

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_{x}\,\mathbf{x}_t + \mathbf{W}_{h}\,\mathbf{h}_{t-1} + \mathbf{b}\right),
\qquad \hat{y} = \mathbf{w}_{\mathrm{out}}^\top\mathbf{h}_{24} + b_{\mathrm{out}}$$

The same $\mathbf{W}_x$, $\mathbf{W}_h$ and $\mathbf{b}$ are used at every hour.
Two things follow.

**The parameter count does not depend on the window length.** A 24-hour window
and a 2400-hour window use the same weights.

**Everything the network remembers is in $\mathbf{h}_t$.** Sixteen numbers
carry the whole past forward, and every hour rewrites them through the $\tanh$,
so what happened early in the window fades as later hours arrive. The LSTM was
built to keep a longer memory.

`nn.RNN` computes the whole loop over the 24 hours in one call and returns
$\mathbf{h}_t$ for every hour. The forecast is read from the last one.

### Your turn

In [ ]:
# TODO 2 --- the recurrent network ------------------------------------------------------------
# One `...` to replace, in forward:
#   out[:, -1, :]      the hidden state after the last hour, h_24, shape (batch, 16)
class RecurrentNet(nn.Module):
    def __init__(self):
        super().__init__()
        # input size  : 1   (one demand value per hour)
        # hidden size : 16  (h_t is a vector of 16 numbers)
        # layers      : 1   (one recurrent layer)
        # parameters  : W_x 16x1 = 16, W_h 16x16 = 256, two bias vectors 16 + 16 = 32  ->  304
        #               (PyTorch keeps two biases where the equation has one)
        self.rnn = nn.RNN(input_size=1, hidden_size=16, num_layers=1, batch_first=True)
        # output size : 1   (the demand one hour ahead)
        # parameters  : w_out 16 + b_out 1  ->  17
        self.head = nn.Linear(16, 1)
        # total       : 304 + 17 = 321, for any window length

    def forward(self, x):                       # x is (batch, 24, 1)
        out, _ = self.rnn(x)                    # out is (batch, 24, 16): h_t for every hour
        h_last = out[:, -1, :]
        return self.head(h_last)                # (batch, 1)
# ------------------------------------------------------------------------------

core.set_seed(0)
rnn = RecurrentNet()
print(f"recurrent network parameters: {core.count_parameters(rnn)}")
print("output shape for 5 windows  :", tuple(rnn(torch.tensor(X_test[:5])).shape))

**What you should see.** `recurrent network parameters: 321` and
`output shape for 5 windows : (5, 1)` — one forecast per window.

---

## 3 · The LSTM

The LSTM adds a second state, the **cell state** $\mathbf{c}_t$, and three
sigmoid **gates** that decide what is kept, what is added and what is shown
(GBC chapter 10, and L5.2):

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(\mathbf{W}_f[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_f)
 && \text{forget gate: what to keep from the cell}\\
\mathbf{i}_t &= \sigma(\mathbf{W}_i[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_i)
 && \text{input gate: how much of the candidate to add}\\
\mathbf{g}_t &= \tanh(\mathbf{W}_g[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_g)
 && \text{the candidate}\\
\mathbf{o}_t &= \sigma(\mathbf{W}_o[\mathbf{x}_t,\mathbf{h}_{t-1}]+\mathbf{b}_o)
 && \text{output gate: how much of the cell to show}\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t\\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t)
\end{aligned}
$$

The line that matters is the one for $\mathbf{c}_t$. It is an **addition**: the
old cell state is kept, scaled by the forget gate, and new information is added
on top. When the forget gate is near one, $\mathbf{c}_t \approx \mathbf{c}_{t-1}$,
and what the cell stored early in the window is carried to the end almost
unchanged. That is how the LSTM keeps a longer memory than the recurrent
network, whose hidden state is rewritten at every hour.

The price is four weight matrices where the recurrent network had one: about four
times the parameters and four times the work per hour.

The cell below has the same shape as the recurrent one. Only the layer changes;
the `forward` is the one you completed in TODO 2.

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self):
        super().__init__()
        # input size  : 1   (one demand value per hour)
        # hidden size : 16  (h_t and the cell state c_t are each 16 numbers)
        # layers      : 1   (one LSTM layer)
        # parameters  : per gate (f, i, g, o): 16x1 + 16x16 + 16 + 16 = 304
        #               four gates: 4 x 304  ->  1,216
        self.lstm = nn.LSTM(input_size=1, hidden_size=16, num_layers=1, batch_first=True)
        # output size : 1   (the demand one hour ahead)
        # parameters  : w_out 16 + b_out 1  ->  17
        self.head = nn.Linear(16, 1)
        # total       : 1,216 + 17 = 1,233, for any window length

    def forward(self, x):                       # x is (batch, 24, 1)
        out, _ = self.lstm(x)                   # out is (batch, 24, 16): h_t for every hour
        h_last = out[:, -1, :]                  # h_24, the hidden state after the last hour
        return self.head(h_last)                # (batch, 1)

core.set_seed(0)
lstm = LSTMNet()
print(f"LSTM parameters             : {core.count_parameters(lstm):,}")
print(f"ratio to the recurrent net  : {core.count_parameters(lstm) / core.count_parameters(rnn):.1f} x")

**What you should see.** `LSTM parameters : 1,233` and a ratio of `3.8 x` —
four gates of 304 parameters each, against one, plus the same 17-parameter head.

---

## 4 · Train both

One recipe for both networks: the full training set in every step, the Adam
optimiser at a learning rate of 0.01, mean squared error, 300 epochs. The
held-out error is recorded after every epoch so that you can see it settle.

### Your turn

In [ ]:
# TODO 3 --- one training step ----------------------------------------------------------------
# Three `...` to replace, inside the loop:
#   line 1  ->  loss_fn(model(Xt), yt)       the training loss on all 702 windows
#   line 2  ->  loss.backward()              gradients of the loss
#   line 3  ->  optimiser.step()             one Adam update
def train(model, epochs=300, lr=0.01):
    loss_fn   = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, yt = torch.tensor(X_train), torch.tensor(y_train)
    Xv, yv = torch.tensor(X_test),  torch.tensor(y_test)
    history = {"train": [], "val": []}
    start = time.time()
    for epoch in range(epochs):
        optimiser.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward()
        optimiser.step()
        with torch.no_grad():
            history["train"].append(float(loss.item()))
            history["val"].append(float(loss_fn(model(Xv), yv).item()))
    history["seconds"] = time.time() - start
    return history
# ------------------------------------------------------------------------------

models = {"recurrent": rnn, "LSTM": lstm}
histories = {}
for name, model in models.items():
    histories[name] = train(model)
    print(f"{name:10s} trained in {histories[name]['seconds']:.1f} s")

fig, ax = plt.subplots(figsize=(7.4, 4.2))
for name, colour in (("recurrent", "#d94f2b"), ("LSTM", "#1f77b4")):
    ax.plot(histories[name]["train"], color=colour, lw=1.2, ls=":", label=f"{name}, training")
    ax.plot(histories[name]["val"],   color=colour, lw=1.8,         label=f"{name}, held out")
ax.axhline(mse_persistence, color="#111111", ls="--", lw=1.2, label="persistence, held out")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel(r"mean squared error [p.u.$^2$]")
ax.set_title("Training both networks")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** Two training times of a few seconds each. On the
machine this was written on they ranged from one to twenty seconds from run to
run, so do not read much into yours.

Then five curves on a log scale. The **recurrent network** drops below the dashed
persistence line after about 70 epochs and then flattens out near 0.0013. The
**LSTM** starts slower: it sits on a plateau for over a hundred epochs, crosses
the persistence line near epoch 145 with a few spikes, and is still falling at
epoch 300, where it ends below the recurrent network. For both, the training and
held-out curves stay close together, so neither network is overfitting.

---

## 5 · The comparison

In [ ]:
rows = []
for name, model in [("persistence", persistence)] + list(models.items()):
    with torch.no_grad():
        pred = model(torch.tensor(X_test)).numpy()
    err = core.mse(pred, y_test)
    rows.append([name, f"{core.count_parameters(model):,}", f"{err:.6f}",
                 f"{err / mse_persistence:.2f}"])
print(core.error_table(rows, ["model", "parameters", "held-out MSE [p.u.^2]",
                              "vs persistence"]))

**What you should see.**

| model | parameters | held-out MSE [p.u.^2] | vs persistence |
| --- | --- | --- | --- |
| persistence | 0 | 0.003320 | 1.00 |
| recurrent | 321 | 0.001298 | 0.39 |
| LSTM | 1,233 | 0.000910 | 0.27 |

The last column is the held-out error divided by the persistence error: below
one is better than persistence.

**Both networks beat persistence**, the recurrent network by a factor of about
2.6 and the LSTM by about 3.6. The problem is learnable and both networks
learned it.

**The LSTM is better, and it paid for it.** Its error is about 70 % of the
recurrent network's, with 3.8 times the parameters and four times the work per
hour. Neither network is fully converged at 300 epochs, so read the ratio as a
snapshot, not a law.

**One run is one run.** Build both networks after `core.set_seed(1)` instead and
the numbers move: the recurrent network ends at about 0.0017 and the LSTM at
about 0.0007. The ranking held at both seeds; the size of the gap did not.

---

## 6 · The forecast, looked at

A table of errors says how large the errors are, not where they happen.

In [ ]:
curves = {}
for name, model in [("persistence", persistence)] + list(models.items()):
    with torch.no_grad():
        curves[name] = model(torch.tensor(X_test)).numpy().ravel()

core.plot_forecast(y_test, curves, n=120,
                   title="The first five days of the held-out period")
plt.show()

**What you should see.** The measured demand in black and three forecasts on
top of it, over 120 hours. The persistence curve is the measured curve shifted
one hour to the right: it is always late, most visibly on the morning rise and
the evening fall. The two networks turn when the measured curve turns, which is
where their advantage in the table comes from. Where they miss, it is mostly at
the peaks and troughs, where they overshoot or undershoot by a few hundredths of
a per unit.

---

## 7 · Save

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb04_sequence.npz")
np.savez(path,
         mse_persistence=mse_persistence,
         names=np.array(list(models)),                                    # recurrent, LSTM
         params=np.array([core.count_parameters(models[n]) for n in models]),
         val=np.array([histories[n]["val"][-1] for n in models]),         # held-out MSE
         train=np.array([histories[n]["train"][-1] for n in models]),     # training MSE
         seconds=np.array([histories[n]["seconds"] for n in models]))
print("wrote", path)
core.saved(path)


**What you should see.** `wrote .../Ex05_outputs/nb04_sequence.npz`.

---

## 8 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The recurrent network and the LSTM both read the same 24-hour window, with a hidden size of 16 and the same training recipe. Say what the hidden state is and how the recurrent network updates it hour by hour. Why does its parameter count of 321 not depend on the window length, and why does what it saw early in the window fade by the last hour?
   *→ L5.2 Q2, Q3, Q4*
2. The LSTM has 1,233 parameters and the recurrent network 321. Account for the difference from the LSTM's equations: name the three gates and the candidate, and say what each one decides. Explain how the addition in the cell-state update lets the LSTM keep a longer memory than the recurrent network. Did the extra parameters pay for themselves on this problem?
   *→ L5.2 Q7, Q9*
3. Persistence has no parameters, and both networks were judged against it. Why is it the right baseline for a one-hour-ahead forecast, and what would a network with a held-out MSE above 0.00332 tell you? In the forecast plot, where does persistence go wrong, and why do the networks do better exactly there?
   *→ L5.2 Q5, Q10*
4. The windows were split in time order and never shuffled. Explain what goes wrong with a shuffled split when two neighbouring windows share twenty-three of their twenty-four hours. Then say what else you would check before claiming that the LSTM is better than the recurrent network, given that changing the seed moved both networks' errors.
   *→ L5.2 Q3, Q10*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.
